# M1.0b Primer: Line, Surface and Volume Integrals

Book pointer: chapter 1, section 1.3 (integration).

Second of four primer notebooks for module 1. Integrals in electromagnetism come in three kinds: along a curve, over a surface, and through a volume. Each is a sum of many small contributions, and each is computed here first as that literal sum, with the formula written out, before any shortcut appears. The code is always the sum, so that the formula and the computation are the same thing. This primer is written to stand on its own: section 2 teaches the ideas, section 3 works each kind of integral by hand and then in NumPy, and sections 6 and 7 collect the results and the warm-up answers. The main notebooks of module 1 (`01_grids_and_operators`, `02_superposition`, `03_potential_and_gradient`) use every one of these integrals on a grid.

## 1. Warm-up

Try on paper first. Answers are in section 7.

1. What does $\int_a^b f(x)\,dx$ mean as a sum? What is the error of a midpoint-rule sum with $N$ pieces, roughly, and what happens to it when $N$ doubles?
2. A wire has mass per length $\lambda(\mathbf{r})$. Write its total mass as an integral along the wire. What plays the role of $dx$?
3. A force $\mathbf{F}$ acts on a particle moving along a path. Write the work done as an integral. Which part of $\mathbf{F}$ contributes, and what happens to the sign if the path is reversed?
4. What is flux? If the velocity of water is $\mathbf{v}$, what does $\int_S \mathbf{v}\cdot d\mathbf{S}$ measure, and in what units?
5. A uniform field $\mathbf{E}$ passes through a flat surface of area $A$ whose normal makes an angle $\theta$ with $\mathbf{E}$. What is the flux?
6. Write the volume element in spherical coordinates and explain where each factor comes from.
7. What does the circle on $\oint$ mean, and why does a closed-surface flux have a sign convention that an open-surface flux does not?

## 2. The ideas

Electromagnetism is built on adding things up. A charged body is a crowd of small charges, and the field it makes is the sum of the fields of the pieces. Moving a charge through a field costs energy that accumulates step by step along the route. The strength of a field through a window is the sum of what passes through every small pane. All three of these are integrals, and the only thing that changes between them is the kind of piece being added: a piece of curve, a piece of surface, or a piece of volume.

Every integral is a limit of a sum. Cut the domain into small pieces, multiply the function on each piece by the size of the piece, add, and refine until the answer stops changing. For a function of one variable the piece is a short interval $\Delta x$, and the sum $\sum f(x_k)\Delta x$ is the familiar area under a curve. On a computer that sum is all we ever compute. The exact integral is what the sum converges to, and the rate of convergence tells us how fine a grid we need. For a smooth integrand the midpoint and trapezoid rules both have errors that shrink as $1/N^2$: doubling the number of pieces cuts the error by four. That number is worth remembering, because it is the same rate that governs every central-difference derivative in notebook 00c and every relaxation solver in module 2.

A curve has a direction, and a piece of curve is a small displacement vector $d\mathbf{l}$, not just a length. Two integrals live on a curve. The scalar one, $\int_C f\,dl$, adds a number times the length of each piece: the mass of a wire, or the total charge on a line of charge with charge per length $\lambda$. The vector one, $\int_C \mathbf{F}\cdot d\mathbf{l}$, first takes the dot product of the field with the step, so only the part of $\mathbf{F}$ along the direction of travel counts. If $\mathbf{F}$ is a force, this is the work done on a particle that follows the curve. Reversing the direction of travel flips every $d\mathbf{l}$ and so flips the sign of the vector integral, while the scalar integral does not care which way you go. When the curve closes on itself the integral is written $\oint_C$, and for a force field its value is the work done in one lap. Fields for which every lap gives zero are called conservative, and the whole of chapter 3 rests on the fact that the electrostatic field is one of them.

A surface has two sides, and a piece of surface is a small vector $d\mathbf{S} = \hat{\mathbf{n}}\,dS$ whose length is the area of the patch and whose direction is the normal. Choosing which of the two normals to use is choosing an orientation. The flux of a field, $\int_S \mathbf{F}\cdot d\mathbf{S}$, adds the part of $\mathbf{F}$ that pokes through each patch times the patch area. For a fluid velocity it is the volume of fluid crossing the surface per second. For the electric field there is nothing flowing, but the picture still works: flux counts how many field lines cross the surface, weighted by direction. A closed surface, written $\oint_S$, has a natural orientation, the outward normal, and with it the flux measures what leaves minus what enters. Gauss' law in chapter 4 says that for the electric field this net outflow is set entirely by the charge inside.

A volume has no direction and a piece of volume is just a number, $dv$. The volume integral $\int_v \rho\,dv$ with $\rho$ the charge per volume is the total charge, and with $\rho = 1$ it is the volume itself. In Cartesian coordinates $dv = dx\,dy\,dz$. In cylindrical and spherical coordinates the pieces are wedges whose sides are not all equal to the coordinate increments: a change $d\phi$ in angle moves you a distance $r\,d\phi$, not $d\phi$. That is where the factors $r$ and $r^2\sin\theta$ in the curvilinear elements come from, and the same factors appear in the surface elements for the same reason. Notebook 00d derives them from scratch; here they are used.

The three integrals are not independent. The divergence theorem turns a volume integral of a certain derivative into a flux through the boundary surface, and Stokes' theorem turns a surface integral of another derivative into a line integral around the boundary curve. Those theorems are the subject of notebook 00d. This notebook is the groundwork: it makes each integral concrete enough that the theorems, when they arrive, are statements about numbers you have already computed.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=5, suppress=True)

### 3.1 One dimension first: integrals as sums

Start with the picture. The integral $\int_a^b f(x)\,dx$ is the signed area between the graph of $f$ and the $x$ axis. A Riemann sum approximates that area by rectangles: cut $[a, b]$ into $N$ pieces of width $\Delta x = (b - a)/N$, pick one point $x_k$ in each piece, and add $f(x_k)\,\Delta x$. As $N$ grows the rectangles hug the curve more closely and the sum converges to the integral. Where you pick $x_k$ inside each piece matters for how fast.

The midpoint rule evaluates $f$ at the centre of each piece. The trapezoid rule evaluates at the two ends and averages, which is the same as replacing the curve by straight segments. Both are exact for a straight line, so their error comes from the curvature of $f$ within each piece, which is proportional to $\Delta x^2$ per piece. Summing $N$ pieces of error $\propto \Delta x^2 \cdot \Delta x$ gives a total error $\propto \Delta x^2 \propto 1/N^2$. Ten times more pieces, a hundred times smaller error. The two rules have opposite signs of error for a curve that bends one way (the midpoint rectangle overshoots where the trapezoid undershoots), and the midpoint error is about half the size.

**Worked example 1.** $\int_0^1 x^2\,dx = 1/3$. With $N = 2$ midpoints at $x = 0.25$ and $0.75$: $(0.0625 + 0.5625)\cdot0.5 = 0.3125$. Error $0.0208$. With $N = 4$ the midpoints are $0.125, 0.375, 0.625, 0.875$ and the sum is $(0.015625 + 0.140625 + 0.390625 + 0.765625)\cdot0.25 = 0.328125$. Error $0.0052$, a quarter of before: doubling $N$ quarters the error.

**Worked example 2 (a non-polynomial integrand).** $\int_0^1 \dfrac{dx}{1 + x^2} = \arctan 1 = \pi/4 = 0.785398$. With $N = 2$: $f(0.25) = 1/1.0625 = 0.941176$, $f(0.75) = 1/1.5625 = 0.64$, sum $\times 0.5 = 0.790588$, error $0.00519$. With $N = 4$: $f(0.125) = 0.984615$, $f(0.375) = 0.876712$, $f(0.625) = 0.719101$, $f(0.875) = 0.566372$, sum $= 3.146800$, times $0.25$ gives $0.786700$, error $0.00130$. The ratio of errors is $3.99$: the $1/N^2$ rule holds for any smooth $f$, not only polynomials.

When the rule fails. The $1/N^2$ rate assumes $f$ is smooth across every piece. A kink, a jump, or a singular slope inside the interval slows the convergence to $1/N$ or worse, because the error is then dominated by the few pieces that straddle the bad point, and refining the grid does not make those pieces behave. The code shows this with $\sqrt{x}$ on $[0, 1]$, whose slope is infinite at $x = 0$: the error falls as $N^{-1.5}$ instead of $N^{-2}$. The same thing happens in three dimensions when a curved boundary is cut by a Cartesian grid (section 3.5): the staircase at the boundary is a kink the grid cannot resolve, and the volume sum converges slowly however smooth the integrand is.

In [ ]:
def midpoint(f, a, b, N):
    xk = a + (np.arange(N) + 0.5) * (b - a) / N
    return np.sum(f(xk)) * (b - a) / N

def trapezoid(f, a, b, N):
    xk = np.linspace(a, b, N + 1)
    return np.trapezoid(f(xk), dx=(b - a) / N)

print("integral of x^2 on [0, 1] = 1/3")
for N in (2, 4, 10, 100, 1000):
    val = midpoint(lambda x: x**2, 0, 1, N)
    print(f"  N = {N:5d}:  midpoint sum = {val:.7f}   error = {abs(val - 1 / 3):.2e}")

print("\nintegral of 1/(1 + x^2) on [0, 1] = pi/4: midpoint and trapezoid")
prev_m = prev_t = None
for N in (2, 4, 8, 16, 32):
    m = midpoint(lambda x: 1 / (1 + x**2), 0, 1, N); t = trapezoid(lambda x: 1 / (1 + x**2), 0, 1, N)
    em, et = abs(m - np.pi / 4), abs(t - np.pi / 4)
    ratio = f"   error ratios {prev_m / em:.2f}, {prev_t / et:.2f}" if prev_m else ""
    print(f"  N = {N:3d}:  midpoint {m:.6f} (err {em:.2e})   trapezoid {t:.6f} (err {et:.2e}){ratio}")
    prev_m, prev_t = em, et

print("\nintegral of sqrt(x) on [0, 1] = 2/3: singular slope at x = 0 slows convergence")
prev = None
for N in (10, 20, 40, 80, 160):
    e = abs(midpoint(np.sqrt, 0, 1, N) - 2 / 3)
    print(f"  N = {N:4d}:  error {e:.2e}" + (f"   ratio {prev / e:.2f}  (4 would be second order, 2.83 is N^-1.5)" if prev else ""))
    prev = e

### 3.2 Line integrals (scalar integrand)

A curve $C$ is described by a parametrisation $\mathbf{r}(t)$ for $t$ from $t_a$ to $t_b$: a recipe that, given a number $t$, returns a point on the curve. The parameter is a label, not a distance. A small change $dt$ moves the point by $d\mathbf{l} = \mathbf{r}'(t)\,dt$, where $\mathbf{r}'(t)$ is the velocity if $t$ were time, and the length of that step is $dl = |\mathbf{r}'(t)|\,dt$. So the arc length of the curve is $\int_C dl = \int_{t_a}^{t_b}|\mathbf{r}'(t)|\,dt$, and the mass of a wire with line density $\lambda$ is $\int_C \lambda\,dl$. A line of charge with charge per length $\lambda$ has total charge $\int_C \lambda\,dl$ in the same way; chapter 2 uses exactly this when it builds the field of a charged wire out of pieces $\lambda\,dl$.

Two parametrisations come up constantly.

- A straight segment from point $\mathbf{A}$ to point $\mathbf{B}$: $\mathbf{r}(t) = \mathbf{A} + t(\mathbf{B} - \mathbf{A})$ for $t \in [0, 1]$. Then $\mathbf{r}' = \mathbf{B} - \mathbf{A}$ is constant and $dl = |\mathbf{B} - \mathbf{A}|\,dt$.
- A circle of radius $R$ in the $xy$ plane: $\mathbf{r}(t) = (R\cos t, R\sin t, 0)$ for $t \in [0, 2\pi]$. Then $\mathbf{r}' = (-R\sin t, R\cos t, 0)$, $|\mathbf{r}'| = R$, and $dl = R\,dt$: the arc length element on a circle is the radius times the angle, which is the scale factor of notebook 00d appearing for the first time.

Pitfall: the parameter is not the arc length unless $|\mathbf{r}'| = 1$. Integrating $\lambda\,dt$ instead of $\lambda\,|\mathbf{r}'|\,dt$ gives a wrong answer whenever the curve is traversed at non-unit speed, which is almost always. The helix below is traversed at constant speed $1.01258$, the parabola at a speed that grows with $x$.

In the computer: sample $t$, build the points, take differences between consecutive points as $\Delta\mathbf{l}$, and sum $\lambda|\Delta\mathbf{l}|$. The differences take care of $|\mathbf{r}'|$ automatically, which is one reason the polyline approach is robust: you never need the derivative in closed form.

**Worked example 3.** One turn of a helix $\mathbf{r}(t) = (\cos t, \sin t, t/2\pi)$, $t \in [0, 2\pi]$: it goes once around a unit circle while rising by one unit. $\mathbf{r}'(t) = (-\sin t, \cos t, 1/2\pi)$, so $|\mathbf{r}'| = \sqrt{1 + 1/4\pi^2} = 1.01258$, constant. Arc length $= 2\pi\cdot1.01258 = 6.3622$. (Flatten the helix out: it is the hypotenuse of a right triangle with legs $2\pi$ and $1$, and $\sqrt{4\pi^2 + 1} = 6.3622$.)

**Worked example 4.** A wire along the parabola $y = x^2$ from $x = 0$ to $1$ with density $\lambda = x$. Parametrise by $x$ itself: $\mathbf{r}(x) = (x, x^2)$, $dl = \sqrt{1 + 4x^2}\,dx$, mass $= \int_0^1 x\sqrt{1 + 4x^2}\,dx = \big[(1 + 4x^2)^{3/2}/12\big]_0^1 = (5^{3/2} - 1)/12 = 0.8484$.

**Worked example 5.** The segment from $\mathbf{A} = (1, 0, 0)$ to $\mathbf{B} = (2, 2, 1)$: $\mathbf{B} - \mathbf{A} = (1, 2, 1)$, length $\sqrt6 = 2.4495$. A ring of radius $R = 2$ carrying a charge per length that varies around the ring as $\lambda(\phi) = \lambda_0\cos^2\phi$ with $\lambda_0 = 1$: total charge $Q = \int_0^{2\pi}\lambda_0\cos^2\phi\;R\,d\phi = \lambda_0R\int_0^{2\pi}\cos^2\phi\,d\phi = \lambda_0R\pi = 2\pi = 6.2832$. The integral of $\cos^2$ over a full period is half the period, because $\cos^2 + \sin^2 = 1$ and the two have the same integral by symmetry. Forgetting the factor $R$ in $dl = R\,d\phi$ would give $\pi$ instead.

In [ ]:
def polyline(r_of_t, t_a, t_b, N=2000):
    t = np.linspace(t_a, t_b, N + 1)
    return r_of_t(t)                                   # (N+1, dim) array of points along the curve

def scalar_line_integral(density, path):
    dl = np.diff(path, axis=0)
    mid = 0.5 * (path[1:] + path[:-1])
    return np.sum(density(mid) * np.linalg.norm(dl, axis=1))

helix = polyline(lambda t: np.stack([np.cos(t), np.sin(t), t / (2 * np.pi)], axis=-1), 0, 2 * np.pi)
print(f"arc length of one helix turn: {scalar_line_integral(lambda p: 1.0, helix):.5f}   exact sqrt(4 pi^2 + 1) = {np.sqrt(4 * np.pi**2 + 1):.5f}")

parabola = polyline(lambda s: np.stack([s, s**2], axis=-1), 0, 1)
print(f"mass of the parabolic wire: {scalar_line_integral(lambda p: p[:, 0], parabola):.5f}   exact (5^1.5 - 1)/12 = {(5**1.5 - 1) / 12:.5f}")

A_pt, B_pt = np.array([1.0, 0.0, 0.0]), np.array([2.0, 2.0, 1.0])
segment = polyline(lambda t: A_pt + t[:, None] * (B_pt - A_pt), 0, 1)
print(f"length of the segment A -> B: {scalar_line_integral(lambda p: 1.0, segment):.5f}   exact sqrt(6) = {np.sqrt(6):.5f}")

R_ring = 2.0
ring = polyline(lambda t: np.stack([R_ring * np.cos(t), R_ring * np.sin(t)], axis=-1), 0, 2 * np.pi)
lam = lambda p: np.cos(np.arctan2(p[:, 1], p[:, 0])) ** 2          # lambda_0 cos^2(phi) with lambda_0 = 1
print(f"charge on the ring with lambda = cos^2(phi): {scalar_line_integral(lam, ring):.5f}   exact pi R = {np.pi * R_ring:.5f}")
print(f"the wrong answer from integrating lambda dphi without the factor R: {np.sum(lam(0.5 * (ring[1:] + ring[:-1]))) * (2 * np.pi / 2000):.5f}")

### 3.3 Line integrals of a vector field: work

When the integrand is a vector field we dot it with the step: $\int_C \mathbf{F}\cdot d\mathbf{l} = \int_{t_a}^{t_b} \mathbf{F}(\mathbf{r}(t))\cdot\mathbf{r}'(t)\,dt$. Only the component of $\mathbf{F}$ along the direction of travel counts; a force at right angles to the motion does no work, however large it is. If $\mathbf{F}$ is a force this is the work done on a particle that moves along $C$. In chapter 3 it is the negative of the change in electric potential.

Sign conventions, because they cause more errors than the integrals do. $W = \int\mathbf{F}\cdot d\mathbf{l}$ is the work done *by* the field on the particle. It is positive when the field pushes the particle the way it is going. The work done *by an external agent* who moves the particle slowly against the field is the negative of that. Reversing the direction of travel reverses every $d\mathbf{l}$ and so reverses the sign: $\int_{B\to A} = -\int_{A\to B}$. When the path returns to its starting point the integral is written with a circle, $\oint_C\mathbf{F}\cdot d\mathbf{l}$, and is called the circulation of $\mathbf{F}$ around $C$.

In general the integral depends on which curve you take between two endpoints. The exception is the most important case in electrostatics: for some fields the integral depends only on the endpoints. Those fields are called conservative, and the test is equivalent to three statements that are each useful:

1. $\oint_C\mathbf{F}\cdot d\mathbf{l} = 0$ around every closed loop.
2. $\int_A^B\mathbf{F}\cdot d\mathbf{l}$ is the same along every path from $A$ to $B$.
3. There is a scalar function $V$ with $\mathbf{F} = -\nabla V$, and then $\int_A^B\mathbf{F}\cdot d\mathbf{l} = V(A) - V(B)$.

Any one implies the others (00d, section 3.6, gives the argument). The third is the one that makes the electric potential possible: if you can find $V$, every line integral of $\mathbf{F}$ becomes a subtraction. Notebook 00c adds a fourth equivalent statement, $\nabla\times\mathbf{F} = 0$.

**Worked example 6.** $\mathbf{F} = (y, x)$ from $(0, 0)$ to $(1, 1)$. Straight line $\mathbf{r}(t) = (t, t)$: $\mathbf{r}' = (1, 1)$, $\mathbf{F}\cdot\mathbf{r}' = t + t = 2t$, integral $\int_0^1 2t\,dt = 1$. Parabola $\mathbf{r}(t) = (t, t^2)$: $\mathbf{r}' = (1, 2t)$, $\mathbf{F} = (t^2, t)$, $\mathbf{F}\cdot\mathbf{r}' = t^2 + 2t^2 = 3t^2$, integral $1$. Same answer. Indeed $(y, x) = \nabla(xy)$ and the integral is $xy$ at the end minus at the start.

**Worked example 7.** $\mathbf{F} = (-y, x)$ along the same two paths. Straight line: $\mathbf{F}\cdot\mathbf{r}' = -t + t = 0$, integral $0$. Parabola: $\mathbf{F} = (-t^2, t)$, $\mathbf{F}\cdot\mathbf{r}' = -t^2 + 2t^2 = t^2$, integral $1/3$. Different answers for the same endpoints: not conservative. Around the unit circle $\mathbf{r}(t) = (\cos t, \sin t)$: $\mathbf{F}\cdot\mathbf{r}' = \sin^2t + \cos^2t = 1$, integral $2\pi$, not zero.

**Worked example 8 (the Coulomb field is conservative).** A charge $q$ at the origin exerts a force $\mathbf{F} = \dfrac{qq_0}{4\pi\epsilon_0}\dfrac{\hat{\mathbf{r}}}{r^2}$ on a test charge $q_0$. Move $q_0$ straight outward along a radius from $r_1$ to $r_2$. Then $d\mathbf{l} = \hat{\mathbf{r}}\,dr$, $\mathbf{F}\cdot d\mathbf{l} = \dfrac{qq_0}{4\pi\epsilon_0}\dfrac{dr}{r^2}$, and
$$W = \frac{qq_0}{4\pi\epsilon_0}\int_{r_1}^{r_2}\frac{dr}{r^2} = \frac{qq_0}{4\pi\epsilon_0}\left(\frac1{r_1} - \frac1{r_2}\right).$$
With $qq_0/4\pi\epsilon_0 = 1$, $r_1 = 1$ and $r_2 = 3$: $W = 1 - 1/3 = 0.6667$. Now take any other path between the same two points. Along a piece of path that moves sideways, $d\mathbf{l}\perp\hat{\mathbf{r}}$ and the field does no work; along a piece that moves radially the contribution is $dr/r^2$ again. So the answer is $1/r_1 - 1/r_2$ for every path: the Coulomb field is conservative, with $V = \dfrac{qq_0}{4\pi\epsilon_0}\dfrac1r$ as the scalar function in statement 3. That is the electric potential energy, and chapter 3 is built on it. The code checks the radial path, a spiral path, and a closed loop.

In [ ]:
def vector_line_integral(F, path):
    '''Sum over steps of F(midpoint) . dl.'''
    dl = np.diff(path, axis=0)
    mid = 0.5 * (path[1:] + path[:-1])
    return np.sum(F(mid) * dl)

F_grad = lambda p: np.stack([p[:, 1], p[:, 0]], axis=-1)        # (y, x) = grad(xy)
F_rot = lambda p: np.stack([-p[:, 1], p[:, 0]], axis=-1)        # (-y, x)

straight = polyline(lambda t: np.stack([t, t], axis=-1), 0, 1)
parab = polyline(lambda t: np.stack([t, t**2], axis=-1), 0, 1)
circle = polyline(lambda t: np.stack([np.cos(t), np.sin(t)], axis=-1), 0, 2 * np.pi)
print("F = (y, x):   straight", round(vector_line_integral(F_grad, straight), 5), "  parabola", round(vector_line_integral(F_grad, parab), 5), "  unit circle", round(vector_line_integral(F_grad, circle), 5))
print("F = (-y, x):  straight", round(vector_line_integral(F_rot, straight), 5), "  parabola", round(vector_line_integral(F_rot, parab), 5), "  unit circle", round(vector_line_integral(F_rot, circle), 5), " (= 2 pi)")

# Worked example 8: the Coulomb force field (q q0 / 4 pi eps0 = 1) along three paths between r = 1 and r = 3.
F_coul = lambda p: p / np.linalg.norm(p, axis=1, keepdims=True) ** 3
radial = polyline(lambda t: np.stack([1 + 2 * t, 0 * t], axis=-1), 0, 1)                                  # straight out along x
spiral = polyline(lambda t: np.stack([(1 + 2 * t) * np.cos(3 * t), (1 + 2 * t) * np.sin(3 * t)], axis=-1), 0, 1)   # same endpoints in r, winding round
spiral = np.vstack([spiral, polyline(lambda t: 3 * np.stack([np.cos(3 + t * (2 * np.pi - 3)), np.sin(3 + t * (2 * np.pi - 3))], axis=-1), 0, 1)])  # finish on the x axis at r = 3
loop = polyline(lambda t: np.stack([2 + np.cos(t), np.sin(t)], axis=-1), 0, 2 * np.pi)
print(f"\nCoulomb field, r = 1 to r = 3:  radial path {vector_line_integral(F_coul, radial):.5f}   spiral path {vector_line_integral(F_coul, spiral):.5f}   exact 1/r1 - 1/r2 = {1 - 1 / 3:.5f}")
print(f"Coulomb field around a closed loop: {vector_line_integral(F_coul, loop):.1e}   (conservative: zero)")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
xg = np.linspace(-0.2, 1.2, 15); Xg, Yg = np.meshgrid(xg, xg, indexing="ij")
for ax, (Fx, Fy), name in zip(axes, ((Yg, Xg), (-Yg, Xg)), ("F = (y, x): path independent", "F = (-y, x): path dependent")):
    ax.quiver(Xg, Yg, Fx, Fy, color="grey", alpha=0.6, pivot="mid")
    ax.plot(straight[:, 0], straight[:, 1], "C0", lw=2, label="straight"); ax.plot(parab[:, 0], parab[:, 1], "C3", lw=2, label="parabola")
    ax.set_aspect("equal"); ax.set_title(name); ax.legend()
plt.show()

**Animation.** The same four line integrals as worked examples 6 and 7, accumulated while a dot walks each path. For $\mathbf{F} = (y, x)$ the straight line and the parabola both finish at 1.000, as they must for a gradient field. For $\mathbf{F} = (-y, x)$ they finish at 0.000 and 0.333, so the path matters. The second half sends a dot around the unit circle: the gradient field's running total swings up and down and returns to zero, while the rotating field's total grows steadily to $2\pi$. The yellow arrow is $\mathbf{F}$ at the dot; the integral grows fastest where that arrow lies along the direction of travel.

In [ ]:
from emlib import show
show.animation("m1_fields", "LineIntegralPaths")

### 3.4 Surface integrals: flux

A surface is parametrised by two variables, $\mathbf{r}(u, v)$: two labels pick a point on the surface, as latitude and longitude pick a point on the globe. A small rectangle $du\,dv$ in parameter space maps to a small parallelogram on the surface with sides $\partial\mathbf{r}/\partial u\,du$ and $\partial\mathbf{r}/\partial v\,dv$. From notebook 00a, the cross product of two sides gives a vector perpendicular to the parallelogram whose length is its area. So

$$d\mathbf{S} = \left(\frac{\partial\mathbf{r}}{\partial u}\times\frac{\partial\mathbf{r}}{\partial v}\right)du\,dv = \hat{\mathbf{n}}\,dS.$$

That one formula gives both the area element and the normal direction. The flux of $\mathbf{F}$ through the surface is $\int_S \mathbf{F}\cdot d\mathbf{S}$, the amount of $\mathbf{F}$ that pokes through, counted positive in the direction of $\hat{\mathbf{n}}$ and negative against it. Field running along the surface contributes nothing, exactly as a force at right angles to a path does no work.

Orientation. Every surface has two sides, and the flux changes sign if you swap them, so a flux is not defined until a normal is chosen. For a closed surface (one that encloses a region: a sphere, a box, a closed cylinder) the convention is always the outward normal, and $\oint_S\mathbf{F}\cdot d\mathbf{S}$ then means net outflow. For an open surface (a disk, a hemisphere, a window) there is no inside, so the orientation must be stated. When the open surface has a boundary curve, the convention that links the two is the right-hand rule: fingers along the direction of travel around the boundary, thumb along $\hat{\mathbf{n}}$. If the cross product formula gives the normal you did not want, swap $u$ and $v$ or flip the sign.

What flux means physically. For a fluid of velocity $\mathbf{v}$, the flux of $\mathbf{v}$ through a surface is the volume of fluid crossing it per unit time: in a short time $dt$ the fluid crossing a patch fills a slanted box of base $dS$ and height $\mathbf{v}\cdot\hat{\mathbf{n}}\,dt$. That is why $\mathbf{v}\cdot\hat{\mathbf{n}}$ and not $|\mathbf{v}|$ appears. For the electric field nothing moves, and $\int\mathbf{E}\cdot d\mathbf{S}$ has units of volt metres, not of anything per second. The picture to keep is field lines: draw them with a density proportional to $|\mathbf{E}|$, and the flux is the net number of lines crossing the surface. A line that enters and leaves a closed surface counts zero, which is the whole reason Gauss' law in chapter 4 works.

**Worked example 9 (uniform field, flat surface).** A uniform field $\mathbf{E} = E_0\hat{\mathbf{z}}$ passes through a flat square of side $a$ whose normal is tilted by an angle $\theta$ from $\hat{\mathbf{z}}$, say $\hat{\mathbf{n}} = (\sin\theta, 0, \cos\theta)$. Both $\mathbf{E}$ and $\hat{\mathbf{n}}$ are constant over the surface, so the integral is a product: flux $= \mathbf{E}\cdot\hat{\mathbf{n}}\,A = E_0a^2\cos\theta$. With $E_0 = 2$, $a = 1.5$, $\theta = 30^\circ$: $2\times2.25\times0.8660 = 3.897$. At $\theta = 90^\circ$ the field skims along the surface and the flux is zero; at $\theta = 180^\circ$ the surface has been turned over and the flux is $-E_0a^2$. Every flux integral is this product, patch by patch, when $\mathbf{E}$ and $\hat{\mathbf{n}}$ vary.

**Worked example 10 (a cylinder).** The radial field $\mathbf{F} = (x, y, 0)$ through the closed surface of a cylinder of radius $R$ and height $H$ around the $z$ axis. On the curved side the outward normal is $\hat{\mathbf{r}}$ (cylindrical) and $\mathbf{F} = R\hat{\mathbf{r}}$ there, so $\mathbf{F}\cdot\hat{\mathbf{n}} = R$ is constant and the flux is $R\times$ area $= R\cdot2\pi RH = 2\pi R^2H$. On the two flat ends the normals are $\pm\hat{\mathbf{z}}$ and $\mathbf{F}$ has no $z$ component: flux zero. With $R = 1$, $H = 2$ the total is $4\pi = 12.566$. The enclosed volume is $\pi R^2H = 2\pi$, so the flux is $2\times$ the volume; the field $(x, y, 0)$ has divergence $2$, and notebook 00d explains why that is not a coincidence.

**Worked example 11 (the sphere's element).** A sphere of radius $R$, parametrised by the polar angle $\theta$ and azimuth $\phi$: $\mathbf{r} = R(\sin\theta\cos\phi, \sin\theta\sin\phi, \cos\theta)$. Then
$\partial\mathbf{r}/\partial\theta = R(\cos\theta\cos\phi, \cos\theta\sin\phi, -\sin\theta)$ and $\partial\mathbf{r}/\partial\phi = R(-\sin\theta\sin\phi, \sin\theta\cos\phi, 0)$. Their cross product works out to $R^2\sin\theta\,(\sin\theta\cos\phi, \sin\theta\sin\phi, \cos\theta) = R^2\sin\theta\,\hat{\mathbf{r}}$. So $dS = R^2\sin\theta\,d\theta\,d\phi$, outward normal $\hat{\mathbf{r}}$. The $\sin\theta$ is there because circles of latitude shrink towards the poles. Integrating $dS$ over the sphere: $R^2\int_0^\pi\sin\theta\,d\theta\int_0^{2\pi}d\phi = R^2\cdot2\cdot2\pi = 4\pi R^2$.

The code checks the cross-product formula against the hand result by finite differences, then builds a general patch routine and uses it for the tilted square, the cylinder and the sphere's area.

In [ ]:
def sphere(theta, phi, R=1.0):
    return R * np.stack([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], axis=-1)

# dS by cross product of finite-difference tangent vectors, at one point.
th, ph, eps = 0.7, 1.1, 1e-6
r_theta = (sphere(th + eps, ph) - sphere(th - eps, ph)) / (2 * eps)
r_phi = (sphere(th, ph + eps) - sphere(th, ph - eps)) / (2 * eps)
dS_vec = np.cross(r_theta, r_phi)
print("dr/dtheta x dr/dphi =", dS_vec)
print("R^2 sin(theta) r_hat =", np.sin(th) * sphere(th, ph))

def surface_patches(r_of_uv, u_range, v_range, nu, nv):
    '''Patch centres and vector area elements dS (both (nu*nv, 3)) from a parametrisation.'''
    u = u_range[0] + (np.arange(nu) + 0.5) * (u_range[1] - u_range[0]) / nu
    v = v_range[0] + (np.arange(nv) + 0.5) * (v_range[1] - v_range[0]) / nv
    du, dv = (u_range[1] - u_range[0]) / nu, (v_range[1] - v_range[0]) / nv
    U, V = np.meshgrid(u, v, indexing="ij")
    e = 1e-6
    ru = (r_of_uv(U + e, V) - r_of_uv(U - e, V)) / (2 * e)
    rv = (r_of_uv(U, V + e) - r_of_uv(U, V - e)) / (2 * e)
    dS = np.cross(ru, rv) * du * dv
    return r_of_uv(U, V).reshape(-1, 3), dS.reshape(-1, 3)

def flux(F, centres, dS):
    return np.sum(F(centres) * dS)

centres, dS = surface_patches(sphere, (0, np.pi), (0, 2 * np.pi), 200, 400)
print(f"\nsphere area from patches: {np.linalg.norm(dS, axis=1).sum():.5f}   exact 4 pi = {4 * np.pi:.5f}")

# Worked example 9: a tilted square of side a, normal (sin theta, 0, cos theta), in the uniform field (0, 0, E0).
E0, a_sq, theta = 2.0, 1.5, np.radians(30)
e1 = np.array([np.cos(theta), 0.0, -np.sin(theta)]); e2 = np.array([0.0, 1.0, 0.0])      # two in-plane directions, e1 x e2 = (sin, 0, cos)
square = lambda u, v: u[..., None] * e1 + v[..., None] * e2
c_sq, d_sq = surface_patches(square, (-a_sq / 2, a_sq / 2), (-a_sq / 2, a_sq / 2), 50, 50)
print(f"uniform field through the tilted square: {flux(lambda p: np.tile([0, 0, E0], (len(p), 1)), c_sq, d_sq):.5f}   exact E0 a^2 cos(theta) = {E0 * a_sq**2 * np.cos(theta):.5f}")

# Worked example 10: radial field (x, y, 0) through a closed cylinder R = 1, H = 2.
R_cyl, H_cyl = 1.0, 2.0
F_rad2 = lambda p: np.stack([p[:, 0], p[:, 1], np.zeros(len(p))], axis=-1)
side = lambda phi, z: np.stack([R_cyl * np.cos(phi), R_cyl * np.sin(phi), z], axis=-1)              # d/dphi x d/dz points outward
c_side, d_side = surface_patches(side, (0, 2 * np.pi), (0, H_cyl), 400, 100)
top = lambda s, phi: np.stack([s * np.cos(phi), s * np.sin(phi), np.full_like(s, H_cyl)], axis=-1)    # d/ds x d/dphi = +z, outward at the top
bottom = lambda phi, s: np.stack([s * np.cos(phi), s * np.sin(phi), np.zeros_like(s)], axis=-1)       # swapped order gives -z, outward at the bottom
c_top, d_top = surface_patches(top, (0, R_cyl), (0, 2 * np.pi), 100, 400); c_bot, d_bot = surface_patches(bottom, (0, 2 * np.pi), (0, R_cyl), 400, 100)
print(f"radial field through the cylinder: side {flux(F_rad2, c_side, d_side):.5f}, top {flux(F_rad2, c_top, d_top):.1e}, bottom {flux(F_rad2, c_bot, d_bot):.1e}   exact 2 pi R^2 H = {2 * np.pi * R_cyl**2 * H_cyl:.5f}")
print(f"check of the normals: mean z of the top normals {np.sign(d_top[:, 2]).mean():+.0f}, of the bottom normals {np.sign(d_bot[:, 2]).mean():+.0f}")

**Worked example 12.** Flux of the radial field $\mathbf{F} = (x, y, z) = \mathbf{r}$ through a sphere of radius $R$. On the sphere $\mathbf{F} = R\hat{\mathbf{r}}$ and $d\mathbf{S} = \hat{\mathbf{r}}R^2\sin\theta\,d\theta\,d\phi$, so $\mathbf{F}\cdot d\mathbf{S} = R^3\sin\theta\,d\theta\,d\phi$ and the flux is $4\pi R^3$. Note that $4\pi R^3 = 3\times\frac43\pi R^3 = 3\times$ volume. Keep that in mind for notebook 00d: the divergence of $\mathbf{r}$ is 3, and worked example 10 showed the same pattern with the factor 2 for the two-dimensional radial field.

**Worked example 13.** Flux of the same field through the upper hemisphere only. By symmetry it is half, $2\pi R^3$. Through the flat disk that closes the hemisphere (at $z = 0$, normal $-\hat{\mathbf{z}}$ to point out of the half-ball), $\mathbf{F}\cdot\hat{\mathbf{n}} = -z = 0$: no flux. So the closed half-ball surface has total flux $2\pi R^3 = 3\times$ its volume. Same pattern.

**Worked example 14.** Flux of the rotation field $(-y, x, 0)$ through the sphere. On the sphere, $\mathbf{F}\cdot\hat{\mathbf{r}} = (-y x + x y + 0)/R = 0$ at every point: the field runs along the surface and never pokes through. Flux zero. A field can be large everywhere on a surface and still have zero flux through it; flux measures crossing, not strength.

In [ ]:
F_radial = lambda p: p
F_rotation = lambda p: np.stack([-p[:, 1], p[:, 0], np.zeros(len(p))], axis=-1)
print(f"radial field through the unit sphere:    {flux(F_radial, centres, dS):.5f}   exact 4 pi = {4 * np.pi:.5f}")
print(f"rotation field through the unit sphere:  {flux(F_rotation, centres, dS):.2e}   exact 0")

hemi_c, hemi_dS = surface_patches(sphere, (0, np.pi / 2), (0, 2 * np.pi), 100, 400)
disk = lambda s, phi: np.stack([s * np.cos(phi), s * np.sin(phi), np.zeros_like(s)], axis=-1)
disk_c, disk_dS = surface_patches(disk, (0, 1), (0, 2 * np.pi), 100, 400)
disk_dS = -disk_dS * np.sign(disk_dS[:, 2:3])            # force the disk normal to point down, out of the half ball
print(f"radial field through the upper hemisphere: {flux(F_radial, hemi_c, hemi_dS):.5f}   through the closing disk: {flux(F_radial, disk_c, disk_dS):.2e}   exact 2 pi = {2 * np.pi:.5f} and 0")

### 3.5 Volume integrals

A volume integral adds a density times small volumes. In Cartesian coordinates the cell is $dv = dx\,dy\,dz$ and the sum runs over a 3D grid. If the density is a charge density $\rho$ (charge per volume), the integral is the total charge. The same sum with $\rho = 1$ gives the volume. The two-dimensional version is an area integral: a sheet with charge per area $\sigma$ has total charge $\int\sigma\,dA$ with $dA = dx\,dy$.

The three charge densities of chapter 2 are the three kinds of integral in this notebook: $\lambda$ (charge per length) is integrated along a curve with $dl$, $\sigma$ (charge per area) over a surface with $dS$, and $\rho$ (charge per volume) through a volume with $dv$. A charged body is always one of these three, and its field is always a sum of point-charge fields weighted by $\lambda\,dl$, $\sigma\,dS$ or $\rho\,dv$. Getting the element right is half of every such calculation.

Curvilinear elements. In cylindrical coordinates the cell between $r$ and $r + dr$, $\phi$ and $\phi + d\phi$, $z$ and $z + dz$ has sides $dr$, $r\,d\phi$ and $dz$, so $dv = r\,dr\,d\phi\,dz$. In spherical coordinates the cell between $r$ and $r + dr$, $\theta$ and $\theta + d\theta$, $\phi$ and $\phi + d\phi$ has sides $dr$, $r\,d\theta$ and $r\sin\theta\,d\phi$, so $dv = r^2\sin\theta\,dr\,d\theta\,d\phi$. In each case the factor in front of the coordinate increments is the product of the scale factors: the distances you actually move per unit change of each coordinate. The surface elements are the same products with one factor left out: $r\,d\phi\,dz$ on the side of a cylinder, $r^2\sin\theta\,d\theta\,d\phi$ on a sphere, $r\,dr\,d\phi$ on a disk. The table in section 6 collects them, and notebook 00d derives the scale factors properly. Pitfall: writing $dv = dr\,d\theta\,d\phi$ is the volume version of forgetting the $R$ in $dl = R\,d\phi$, and it is the most common error in these integrals.

**Worked example 15 (a sheet).** A square sheet $[0, a]\times[0, a]$ with charge per area $\sigma = \sigma_0(1 + x/a)$, denser towards the far edge. $Q = \int_0^a\!\int_0^a\sigma_0(1 + x/a)\,dx\,dy = \sigma_0a\big[x + x^2/2a\big]_0^a = \sigma_0a(a + a/2) = \tfrac32\sigma_0a^2$. With $\sigma_0 = 1$, $a = 2$: $Q = 6$. The average density is $1.5\sigma_0$, as it must be for a density that runs linearly from $\sigma_0$ to $2\sigma_0$.

**Worked example 16 (a cylinder).** A solid cylinder of radius $R$ and height $H$ with $\rho = \rho_0r/R$, zero on the axis and densest at the rim. Using the cylindrical element: $Q = \int_0^H\!\int_0^{2\pi}\!\int_0^R\rho_0\frac rR\;r\,dr\,d\phi\,dz = \rho_0\frac1R\cdot\frac{R^3}3\cdot2\pi\cdot H = \tfrac23\pi\rho_0R^2H$. With $\rho_0 = R = 1$, $H = 2$: $Q = 4\pi/3 = 4.1888$. Without the factor $r$ in the element the answer would be $\rho_0\frac{R^2}{2R}\cdot2\pi H = 2\pi$, wrong by a factor $2/3$.

**Worked example 17 (a ball).** A ball of radius $R$ with charge density $\rho = \rho_0(1 - r/R)$, densest at the centre and zero at the surface. Total charge in spherical coordinates, using the angular integrals $\int\sin\theta\,d\theta\,d\phi = 4\pi$:
$$Q = 4\pi\rho_0\int_0^R \left(1 - \frac rR\right)r^2\,dr = 4\pi\rho_0\left(\frac{R^3}{3} - \frac{R^3}{4}\right) = \frac{\pi\rho_0R^3}{3}.$$
With $\rho_0 = R = 1$: $Q = 1.0472$. The code does the same integral two ways: as a Cartesian grid sum over a cube, keeping only cells inside the ball, and as a spherical grid sum where the $r^2\sin\theta$ factor appears explicitly.

In [ ]:
# Worked example 15: the sheet, as a 2D midpoint sum.
a_sheet, n2 = 2.0, 200
s2 = (np.arange(n2) + 0.5) * (a_sheet / n2)
Xs, Ys = np.meshgrid(s2, s2, indexing="ij")
Q_sheet = np.sum(1 + Xs / a_sheet) * (a_sheet / n2) ** 2
print(f"sheet with sigma = 1 + x/a on [0, 2]^2:  Q = {Q_sheet:.5f}   exact 1.5 sigma0 a^2 = {1.5 * a_sheet**2:.5f}")

# Worked example 16: the cylinder with rho = r/R, on a cylindrical grid with dv = r dr dphi dz.
R_c, H_c, nr, nphi, nz = 1.0, 2.0, 100, 100, 50
rc = (np.arange(nr) + 0.5) * R_c / nr; phc = (np.arange(nphi) + 0.5) * 2 * np.pi / nphi; zc = (np.arange(nz) + 0.5) * H_c / nz
Rg, Pg, Zg = np.meshgrid(rc, phc, zc, indexing="ij")
dv_cyl = Rg * (R_c / nr) * (2 * np.pi / nphi) * (H_c / nz)
print(f"cylinder with rho = r/R:  Q = {np.sum(Rg / R_c * dv_cyl):.5f}   exact 2 pi R^2 H / 3 = {2 * np.pi * R_c**2 * H_c / 3:.5f}")
print(f"  the same sum without the factor r in dv: {np.sum(Rg / R_c) * (R_c / nr) * (2 * np.pi / nphi) * (H_c / nz):.5f}  (wrong)")

# Worked example 17: the ball.
rho0, R = 1.0, 1.0
rho = lambda r: rho0 * (1 - r / R)

# Cartesian: cells of a cube, keep those inside the ball.
for n in (20, 40, 80, 160):
    s = -R + (np.arange(n) + 0.5) * (2 * R / n)
    Xc, Yc, Zc = np.meshgrid(s, s, s, indexing="ij")
    rc3 = np.sqrt(Xc**2 + Yc**2 + Zc**2)
    Q_cart = np.sum(np.where(rc3 <= R, rho(rc3), 0.0)) * (2 * R / n) ** 3
    print(f"ball, Cartesian grid, {n}^3 cells:  Q = {Q_cart:.5f}   error {abs(Q_cart - np.pi / 3):.1e}")

# The staircase on its own: the volume of the ball (rho = 1) by the same Cartesian count.
for n in (20, 40, 80, 160):
    s = -R + (np.arange(n) + 0.5) * (2 * R / n)
    Xc, Yc, Zc = np.meshgrid(s, s, s, indexing="ij")
    V_cart = np.sum(Xc**2 + Yc**2 + Zc**2 <= R**2) * (2 * R / n) ** 3
    print(f"ball volume, Cartesian grid, {n}^3 cells:  V = {V_cart:.5f}   error {abs(V_cart - 4 * np.pi / 3):.1e}")

# Spherical: cells in (r, theta, phi) with volume r^2 sin(theta) dr dtheta dphi. No staircase, so the error falls as 1/N^2 for both Q and the volume.
for nr, nt, np_ in ((25, 12, 24), (50, 25, 50), (100, 50, 100)):
    r = (np.arange(nr) + 0.5) * R / nr; th = (np.arange(nt) + 0.5) * np.pi / nt; ph = (np.arange(np_) + 0.5) * 2 * np.pi / np_
    Rg, Tg, Pg = np.meshgrid(r, th, ph, indexing="ij")
    dv = Rg**2 * np.sin(Tg) * (R / nr) * (np.pi / nt) * (2 * np.pi / np_)
    print(f"ball, spherical grid {nr}x{nt}x{np_}:  Q = {np.sum(rho(Rg) * dv):.5f}   error {abs(np.sum(rho(Rg) * dv) - np.pi / 3):.1e}   volume {np.sum(dv):.5f}   (exact pi/3 = {np.pi / 3:.5f}, 4 pi/3 = {4 * np.pi / 3:.5f})")

Read the Cartesian results carefully. For this $\rho$, which vanishes at the surface, the staircase hardly matters, because the cells that straddle the sphere carry almost no charge; the error is small but it does not fall by a steady factor of four per doubling. The volume count with $\rho = 1$ shows the staircase on its own: the error jumps about from one $n$ to the next and is set by how many cell centres happen to land just inside or just outside the sphere, not by the smoothness of the integrand. That is the three-dimensional version of the $\sqrt{x}$ example in section 3.1, a feature the grid cannot resolve limiting the convergence. The spherical grid fits the ball exactly, and its errors in both $Q$ and the volume fall cleanly by a factor of four per doubling, the $1/N^2$ rate of section 3.1. That is the general lesson of section 1.5 of the book and of notebook 00d: match the coordinates to the geometry. When the geometry has no matching coordinates, which is the usual case in the main notebooks, accept the staircase and refine until the answer stops moving.

### Exercise: flux through a cube

Replace the body of `flux_through_cube` so that it computes the outward flux of a field through the six faces of the cube $|x|, |y|, |z| \le a$ by sampling each face on a grid. Each face has a constant outward normal ($\pm\hat{\mathbf{x}}$, $\pm\hat{\mathbf{y}}$ or $\pm\hat{\mathbf{z}}$), so the flux through it is the integral of one component of $\mathbf{F}$ over the face, with the sign of the normal. The check uses the radial field, whose flux through any closed surface is $3\times$ the enclosed volume, here $3\cdot(2a)^3 = 24a^3$.

In [ ]:
def flux_through_cube(F, a, n=200):
    '''Outward flux of F through the surface of the cube |x|,|y|,|z| <= a.'''
    s = -a + (np.arange(n) + 0.5) * (2 * a / n)                 # replace from here with your own version
    U, V = np.meshgrid(s, s, indexing="ij")
    dA = (2 * a / n) ** 2
    total = 0.0
    for axis in range(3):
        for sign in (+1.0, -1.0):
            pts = np.zeros((n * n, 3))
            others = [k for k in range(3) if k != axis]
            pts[:, others[0]] = U.ravel(); pts[:, others[1]] = V.ravel(); pts[:, axis] = sign * a
            total += sign * np.sum(F(pts)[:, axis]) * dA
    return total


for a in (0.5, 1.0):
    val = flux_through_cube(F_radial, a)
    print(f"a = {a}:  flux of (x, y, z) through the cube = {val:.5f}   exact 24 a^3 = {24 * a**3}")
    assert abs(val - 24 * a**3) < 1e-6

## 4. Verify

A field whose flux you can compute by hand through two different closed surfaces: the Coulomb field $\mathbf{F} = \hat{\mathbf{r}}/r^2$, the shape of the field of a point charge at the origin. Through a sphere of radius $R$ centred on the origin, $\mathbf{F}\cdot d\mathbf{S} = (1/R^2)\,R^2\sin\theta\,d\theta\,d\phi$ and the flux is $4\pi$, whatever $R$ is: the field weakens as $1/R^2$ and the area grows as $R^2$, and the two cancel exactly. Through the unit cube centred on the origin the integral is harder by hand, but the answer is also $4\pi$. Through any closed surface that does not enclose the origin, it is zero: every field line that enters must leave. Chapter 4 turns this fact into Gauss' law. The cube and the displaced sphere are checked numerically here; the displaced sphere needs a fine grid near the side closest to the origin, where the field is strongest and changes fastest.

In [ ]:
F_coulomb = lambda p: p / np.linalg.norm(p, axis=1, keepdims=True) ** 3
for R in (0.5, 1.0, 2.0):
    c, d = surface_patches(lambda t, p: sphere(t, p, R), (0, np.pi), (0, 2 * np.pi), 200, 400)
    print(f"sphere radius {R}: flux = {flux(F_coulomb, c, d):.5f}", end="   ")
print(f"\nunit cube:            flux = {flux_through_cube(F_coulomb, 1.0, n=400):.5f}   4 pi = {4 * np.pi:.5f}")
shifted = lambda t, p: sphere(t, p, 1.0) + np.array([3.0, 0.0, 0.0])
c, d = surface_patches(shifted, (0, np.pi), (0, 2 * np.pi), 300, 600)
print(f"unit sphere centred at (3, 0, 0), origin outside: flux = {flux(F_coulomb, c, d):.2e}")
assert abs(flux(F_coulomb, c, d)) < 1e-3 and abs(flux_through_cube(F_coulomb, 1.0, n=400) - 4 * np.pi) < 1e-3

## 5. Explore and rebuild

Explore. The work done by $\mathbf{F} = (-y, x)$ around a closed loop came out as $2\pi$ for the unit circle. Compute it for circles of radius $0.5$, $1$, $2$ and for the unit square. Divide each result by the enclosed area. The ratio is the same number every time. Which number, and which of the six reference fields from notebook 00a has that number as its curl? That is Stokes' theorem waiting to be stated in notebook 00d.

Rebuild. Close this notebook. On paper, derive $d\mathbf{S}$ for the sphere from the cross product of the two tangent vectors (worked example 11), derive the total charge of the ball in worked example 17, and write out the Coulomb work integral of worked example 8. In a blank editor, write `vector_line_integral`, `surface_patches` and `flux` from scratch and reproduce the $4\pi$ results.

## 6. Key results

| result | formula | holds when |
|---|---|---|
| Riemann sum | $\int_a^b f\,dx \approx \sum_k f(x_k)\,\Delta x$ | always; converges as $N\to\infty$ |
| midpoint and trapezoid error | $\propto 1/N^2$ | $f$ smooth on every piece; $\propto 1/N$ or $N^{-1.5}$ with kinks, singular slopes, staircased boundaries |
| line element | $d\mathbf{l} = \mathbf{r}'(t)\,dt$, $dl = \lvert\mathbf{r}'(t)\rvert\,dt$ | any parametrisation; $dl \ne dt$ unless $\lvert\mathbf{r}'\rvert = 1$ |
| segment $A\to B$ | $\mathbf{r} = \mathbf{A} + t(\mathbf{B} - \mathbf{A})$, $dl = \lvert\mathbf{B}-\mathbf{A}\rvert\,dt$ | $t\in[0,1]$ |
| circle of radius $R$ | $\mathbf{r} = (R\cos t, R\sin t)$, $dl = R\,dt$ | $t\in[0, 2\pi]$ |
| scalar line integral | $\int_C \lambda\,dl$ | mass or charge of a wire; independent of direction |
| work | $W = \int_C \mathbf{F}\cdot d\mathbf{l} = \int \mathbf{F}\cdot\mathbf{r}'\,dt$ | sign flips with direction; only the along-path part of $\mathbf{F}$ counts |
| conservative field | $\oint\mathbf{F}\cdot d\mathbf{l} = 0 \iff$ path independent $\iff \mathbf{F} = -\nabla V$ | then $\int_A^B\mathbf{F}\cdot d\mathbf{l} = V(A) - V(B)$ |
| Coulomb work, radial | $W = \dfrac{qq_0}{4\pi\epsilon_0}\left(\dfrac1{r_1} - \dfrac1{r_2}\right)$ | any path between radii $r_1$ and $r_2$ |
| surface element | $d\mathbf{S} = \left(\partial_u\mathbf{r}\times\partial_v\mathbf{r}\right)du\,dv = \hat{\mathbf{n}}\,dS$ | sign fixed by orientation (outward for closed surfaces) |
| flux | $\int_S\mathbf{F}\cdot d\mathbf{S}$ | uniform $\mathbf{F}$ through a flat surface: $\mathbf{F}\cdot\hat{\mathbf{n}}A = FA\cos\theta$ |
| sphere | $dS = R^2\sin\theta\,d\theta\,d\phi$, $\hat{\mathbf{n}} = \hat{\mathbf{r}}$, area $4\pi R^2$ | |
| cylinder side | $dS = R\,d\phi\,dz$, $\hat{\mathbf{n}} = \hat{\mathbf{r}}$ (cylindrical), area $2\pi RH$ | |
| flux of $\mathbf{r}$ | $4\pi R^3 = 3\times$ volume through a sphere; $2\times$ volume for $(x,y,0)$ through a cylinder | previews divergence $= 3$ and $2$ |
| flux of $\hat{\mathbf{r}}/r^2$ | $4\pi$ through any closed surface around the origin, $0$ otherwise | Gauss' law in chapter 4 |
| volume integral | $\int_v\rho\,dv$ | total charge; volume when $\rho = 1$ |

Elements in the three coordinate systems (derived in notebook 00d):

| system | coordinates | $dl$ along each coordinate | $dv$ | common $dS$ |
|---|---|---|---|---|
| Cartesian | $(x, y, z)$ | $dx,\ dy,\ dz$ | $dx\,dy\,dz$ | $dx\,dy$ on a plane $z = $ const |
| cylindrical | $(r, \phi, z)$ | $dr,\ r\,d\phi,\ dz$ | $r\,dr\,d\phi\,dz$ | $r\,d\phi\,dz$ on a cylinder, $r\,dr\,d\phi$ on a disk |
| spherical | $(r, \theta, \phi)$ | $dr,\ r\,d\theta,\ r\sin\theta\,d\phi$ | $r^2\sin\theta\,dr\,d\theta\,d\phi$ | $r^2\sin\theta\,d\theta\,d\phi$ on a sphere |

Useful angular integrals: $\int_0^{2\pi}\cos^2\phi\,d\phi = \pi$, $\int_0^\pi\sin\theta\,d\theta = 2$, $\int_0^\pi\sin\theta\cos\theta\,d\theta = 0$, $\int_0^\pi\sin\theta\cos^2\theta\,d\theta = 2/3$, and $\int\sin\theta\,d\theta\,d\phi = 4\pi$ over the whole sphere.

## 7. Warm-up answers

1. It is the limit of $\sum_k f(x_k)\,\Delta x$ as the pieces shrink. For a smooth $f$ the midpoint-rule error is proportional to $1/N^2$, so doubling $N$ divides the error by four.
2. $M = \int_C\lambda\,dl$ with $dl = |\mathbf{r}'(t)|\,dt$. The arc length element $dl$ plays the role of $dx$; the parameter increment $dt$ does not, unless the curve is traversed at unit speed.
3. $W = \int_C\mathbf{F}\cdot d\mathbf{l}$. Only the component of $\mathbf{F}$ along the direction of travel contributes. Reversing the path reverses every $d\mathbf{l}$ and flips the sign of $W$.
4. Flux is the surface integral of the normal component of a vector field. For water, $\int_S\mathbf{v}\cdot d\mathbf{S}$ is the volume of water crossing $S$ per unit time, in m$^3$/s.
5. $\Phi = EA\cos\theta$. It is $EA$ when the field is normal to the surface, zero when it skims along it, and negative when the normal points against the field.
6. $dv = r^2\sin\theta\,dr\,d\theta\,d\phi$. The three factors are the sides of the small cell: $dr$ radially, $r\,d\theta$ along a meridian (an arc of radius $r$), and $r\sin\theta\,d\phi$ along a circle of latitude (an arc of radius $r\sin\theta$).
7. The circle marks an integral over a closed curve or closed surface. A closed surface encloses a region, so "outward" is defined and is the convention; an open surface has two equally good normals and the orientation must be chosen and stated.